<a href="https://colab.research.google.com/github/jalmendral/BA305-A1-TeamA2/blob/main/notebooks/00_cleaning_Grant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# --- bootstrap: Colab only (hardcoded /content path) ---
import subprocess
from pathlib import Path
import pandas as pd, numpy as np

REPO_DIR = Path("/content/BA305-A1-TeamA2")
RAW  = REPO_DIR / "Data" / "raw"
PROC = REPO_DIR / "Data" / "processed"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "-q",
         "https://github.com/jalmendral/BA305-A1-TeamA2.git", str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)

print(sorted(p.name for p in RAW.iterdir()))   # confirms the clone worked

['all_recipes.csv', 'cuisines.csv']


In [15]:
# cleaning notebook only
RAW = REPO_DIR / "Data" / "raw"
raw_a = pd.read_csv(RAW / "all_recipes.csv")
raw_c = pd.read_csv(RAW / "cuisines.csv")

In [16]:
# --- Grant: data cleaning, union, outlier rules, missingness flags ---

# Union, not join. 1,163 cuisine recipes appear nowhere in all_recipes.
df = pd.concat([raw_a, raw_c[~raw_c.url.isin(raw_a.url)]], ignore_index=True)
df["country"]     = df.url.map(raw_c.set_index("url").country)
df["has_cuisine"] = df.country.notna()
assert df.url.is_unique

df["year"] = pd.to_datetime(df.date_published).dt.year

# Implausible per-serving nutrition becomes NaN and is flagged. Not deleted,
# not filled in. Filling happens per fold in the modelling notebooks (§5.3).
limits = {"calories": 2000, "fat": 200, "carbs": 300, "protein": 100}
df["nutrition_flag"] = False
for col, cap in limits.items():
    bad = df[col] > cap
    print(f"{col}: {bad.sum()} implausible values set to NaN")
    df["nutrition_flag"] |= bad
    df.loc[bad, col] = np.nan

df["time_flag"] = (df.total_time == 0) | (df.total_time < df.prep_time + df.cook_time)
print(f"time_flag: {df.time_flag.sum()} rows")
df.loc[df.time_flag, ["prep_time", "cook_time", "total_time"]] = np.nan

df["rating_missing"]    = df.avg_rating.isna()
df["nutrition_missing"] = df[list(limits)].isna().any(axis=1)

df["ingredients"] = df.ingredients.fillna("").str.lower().str.strip()

print(df.shape, "|", df.has_cuisine.sum(), "with cuisine")

calories: 21 implausible values set to NaN
fat: 9 implausible values set to NaN
carbs: 10 implausible values set to NaN
protein: 18 implausible values set to NaN
time_flag: 406 rows
(15589, 23) | 2218 with cuisine


In [17]:
df.groupby("year").rating_missing.agg(["mean", "size"]).round(3)
df.groupby("year").nutrition_missing.mean().round(3)

,nutrition_missing
year,
2004,0.000
2006,0.000
2007,0.000
2008,0.000
2009,0.000
2010,0.000
2011,0.000
2012,0.000
2013,0.000


In [18]:
df.to_csv("recipes_clean.csv", index=False)
from google.colab import files
files.download("recipes_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>